# Fish Toxicty Further Algorithms & Scaling
In this dataset, each row is an experiment on the rainbow trout. The columns show what chemicals were present, how long the fish were left in the toxic water, the temperature of the water, etc. With that, our target variable is "result_conc1_mean_mol_log" which is the The base-10 logarithm of the mean molar concentration log10(mol/L) required to reach the specified experimental toxicity endpoint (LC50)

In [4]:
import pandas as pd

df = pd.read_csv(
    "../adore_dataset/processed/s-F2F-1_mortality.csv",
    dtype=str
)
# features and targets
features = df[[
    "chem_mw",
    "chem_rdkit_clogp",
    "chem_pcp_heavy_atom_count",
    "chem_pcp_bonds_count",
    "chem_rings_count",
    "chem_OH_count",
    #"media_ph_mean",           
    #"media_temperature_mean",
    "result_obs_duration_mean",
    "test_exposure_type"
]]


target = df["result_conc1_mean_mol_log"]

X, y = features, target
numeric_cols = [
    'chem_mw',
    'chem_rdkit_clogp',
    'chem_pcp_heavy_atom_count',
    'chem_pcp_bonds_count',
    'chem_rings_count',
    'chem_OH_count',
    'result_obs_duration_mean'
]

X[numeric_cols] = X[numeric_cols].apply(pd.to_numeric)

X = pd.get_dummies(X, columns=['test_exposure_type'], drop_first=True)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
y_train = pd.to_numeric(y_train, errors="coerce")
y_test = pd.to_numeric(y_test, errors="coerce")

In [5]:
# Linear models
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet

# Tree-based models
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

# Other regression models
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

# Model evaluation
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Train/test split
from sklearn.model_selection import train_test_split

# Scaling (important for Ridge, Lasso, ElasticNet, KNN, and SVR)
from sklearn.preprocessing import StandardScaler

In [6]:


linear = LinearRegression()

lasso = Lasso(alpha=1.0)

elastic_net = ElasticNet(alpha=1.0)

decision_tree = DecisionTreeRegressor(random_state=42)

random_forest = RandomForestRegressor(n_estimators=100, random_state=42)

extra_trees = ExtraTreesRegressor(n_estimators=100, random_state=42)

gradient_boosting = GradientBoostingRegressor(random_state=42)

hist_gradient_boosting = HistGradientBoostingRegressor(random_state=42)

knn = KNeighborsRegressor(n_neighbors=5)

svr = SVR()


# Dictionary containing all models
models = {
    "Linear Regression": linear,
    "Lasso": lasso,
    "Elastic Net": elastic_net,
    "Decision Tree": decision_tree,
    "Random Forest": random_forest,
    "Extra Trees": extra_trees,
    "Gradient Boosting": gradient_boosting,
    "Hist Gradient Boosting": hist_gradient_boosting,
    "KNN": knn,
    "SVR": svr
}

In [10]:
from sklearn.model_selection import cross_val_score, KFold
cv = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv)
    print(f"{name}: {scores.mean():.3f} ± {scores.std():.3f}")

Linear Regression: 0.523 ± 0.019
Lasso: 0.428 ± 0.007
Elastic Net: 0.440 ± 0.009
Decision Tree: 0.794 ± 0.031
Random Forest: 0.852 ± 0.019
Extra Trees: 0.846 ± 0.021
Gradient Boosting: 0.748 ± 0.024
Hist Gradient Boosting: 0.831 ± 0.018
KNN: 0.682 ± 0.034
SVR: 0.489 ± 0.009


In [ ]:
# KNN, SVR, Ridge, Lasso, and Elastic Net are sensitive to feature scale, 
# so we should ideally add scaling before comparing them fairly.

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(name)
    print("R²:", r2_score(y_test, y_pred))
    print("MAE:", mean_absolute_error(y_test, y_pred))
    print("MSE:", mean_squared_error(y_test, y_pred))
    print("-----------------------")

Linear Regression
R²: 0.5738054542542245
MAE: 0.80153654940017
MSE: 1.0985260033418405
-----------------------
Lasso
R²: 0.4849865773723663
MAE: 0.8954684035970611
MSE: 1.3274586511578896
-----------------------
Elastic Net
R²: 0.4973908399209941
MAE: 0.8791938637010541
MSE: 1.2954863861489536
-----------------------
Decision Tree
R²: 0.8171908095701509
MAE: 0.40799955590675235
MSE: 0.47119478966032763
-----------------------
Random Forest
R²: 0.8587775657232934
MAE: 0.38172369860520555
MSE: 0.3640039926759995
-----------------------
Extra Trees
R²: 0.8555561955748208
MAE: 0.3773172816991655
MSE: 0.37230714650518354
-----------------------
Gradient Boosting
R²: 0.7631867861631544
MAE: 0.5721961276001359
MSE: 0.6103913715730728
-----------------------
Hist Gradient Boosting
R²: 0.836720588473439
MAE: 0.43414632935042696
MSE: 0.42085634638617053
-----------------------
KNN
R²: 0.7281998082349447
MAE: 0.5663318876998253
MSE: 0.7005710921164963
-----------------------
SVR
R²: 0.53619737678